# M04 — CTA Data Cleaning

M02 ki filtered bus aur rail files ko next phases ke liye prepare karein. Dono datasets separate rahenge. Original files change nahi hongi. Cells upar se neeche run karein.

**Scope:** basic cleaning only. Features, target labels, train/test split aur ML model is notebook ka hissa nahi hain.


## 1. Libraries aur data folder

Notebook project root, `backend` ya `backend/notebooks` se run ho sakti hai. Pehle M02 ki filtered CSVs `backend/data/processed` mein honi chahiye.


In [9]:
import pandas as pd
from pathlib import Path

if Path("backend/data/processed").is_dir():
    data_folder = Path("backend/data/processed")
elif Path("data/processed").is_dir():
    data_folder = Path("data/processed")
else:
    data_folder = Path("../data/processed")

print("Data folder:", data_folder.resolve())


Data folder: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\data\processed


## 2. Bus aur rail data load karein

Identifiers text ki tarah read hote hain, taa ke unka format preserve ho. Hum memory mein copies par kaam karte hain.


In [10]:
bus = pd.read_csv(data_folder / "bus_2022_2025.csv", dtype=str, keep_default_na=False)
rail = pd.read_csv(data_folder / "rail_2022_2025.csv", dtype=str, keep_default_na=False)
datasets = {"Bus": bus, "Rail": rail}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows | columns: {list(df.columns)}")


Bus: 167,648 rows | columns: ['route', 'date', 'daytype', 'rides']
Rail: 209,472 rows | columns: ['station_id', 'stationname', 'date', 'daytype', 'rides']


## 3. Column names, spaces aur blank values

Required columns check karte hain. Text ke shuru/end ki spaces remove hoti hain; empty values missing mark hoti hain. Ye changes original CSV mein save nahi hotin.


In [11]:
required = {
    "Bus": {"route", "date", "daytype", "rides"},
    "Rail": {"station_id", "stationname", "date", "daytype", "rides"},
}

for name, df in datasets.items():
    missing_columns = required[name] - set(df.columns)
    if missing_columns:
        raise ValueError(f"{name}: required columns missing: {sorted(missing_columns)}")
    datasets[name] = df.apply(lambda column: column.str.strip().replace("", pd.NA))
    print(f"\n{name} missing values:")
    print(datasets[name].isna().sum())



Bus missing values:
route      0
date       0
daytype    0
rides      0
dtype: int64

Rail missing values:
station_id     0
stationname    0
date           0
daytype        0
rides          0
dtype: int64


## 4. Dates aur rides ko correct types dein

Dates ko datetime aur rides ko numeric banate hain. Unreadable values missing (`NaT` / `NaN`) banengi; next cell unhein check karega. Rides ka valid non-negative integer hona zaroori hai.


In [12]:
for name, df in datasets.items():
    df["date"] = pd.to_datetime(
        df["date"], format="mixed", errors="coerce"
    )

    # Valid thousands separators ko remove karo.
    rides_text = df["rides"]
    has_thousands_separator = rides_text.str.fullmatch(
        r"\d{1,3}(?:,\d{3})+", na=False
    )
    cleaned_rides = rides_text.where(
        ~has_thousands_separator,
        rides_text.str.replace(",", "", regex=False)
    )

    df["rides"] = pd.to_numeric(cleaned_rides, errors="coerce")

    print(f"{name}: date = {df['date'].dtype}, rides = {df['rides'].dtype}")

Bus: date = datetime64[ns], rides = int64
Rail: date = datetime64[ns], rides = int64


## 5. Invalid values check karein

Route/station ID, date, day type aur rides essential hain. Agar invalid records hon, cleaning rukegi: pehle un records ko investigate karein. Is notebook mein missing values guess karke fill ya problematic rows automatically delete nahi hotin. Counts overlap kar sakte hain.


In [13]:
ready_to_save = False
problems_found = False
for name, df in datasets.items():
    id_column = "route" if name == "Bus" else "station_id"
    invalid_rides = df["rides"].isna() | df["rides"].isin([float("inf"), float("-inf")])
    invalid_rides |= (df["rides"] < 0) | (df["rides"] % 1 != 0)
    checks = {
        "Missing route/station ID": df[id_column].isna(),
        "Missing or invalid date": df["date"].isna(),
        "Date outside 2022–2025": df["date"].notna() & ~df["date"].between("2022-01-01", "2025-12-31 23:59:59.999999999"),
        "Missing or unexpected day type": ~df["daytype"].isin(["W", "A", "U"]),
        "Missing or invalid rides": invalid_rides,
    }
    print(f"\n{name} checks:")
    for label, mask in checks.items():
        count = int(mask.sum())
        print(f"  {label}: {count}")
        if count:
            problems_found = True

if problems_found:
    raise ValueError("Invalid data mila hai. Save se pehle M03 results aur affected rows investigate karein.")



Bus checks:
  Missing route/station ID: 0
  Missing or invalid date: 0
  Date outside 2022–2025: 0
  Missing or unexpected day type: 0
  Missing or invalid rides: 0

Rail checks:
  Missing route/station ID: 0
  Missing or invalid date: 0
  Date outside 2022–2025: 0
  Missing or unexpected day type: 0
  Missing or invalid rides: 0


## 6. Exact duplicate rows remove karein

Puri row repeat ho to pehli copy rakhi jati hai. Dates ko daily format aur rides ko integer banaya jata hai. **Zero rides valid reh sakti hain**, isliye unhein remove nahi karte.


In [14]:
for name, df in datasets.items():
    df["date"] = df["date"].dt.normalize()
    df["rides"] = df["rides"].astype("int64")
    duplicates = int(df.duplicated().sum())
    datasets[name] = df.drop_duplicates().copy()
    print(f"{name}: {duplicates} exact duplicate rows removed; {len(datasets[name]):,} rows remain")


Bus: 0 exact duplicate rows removed; 167,648 rows remain
Rail: 0 exact duplicate rows removed; 209,472 rows remain


## 7. Same route/station aur date ki remaining duplicates

Agar ek hi route/station aur date ki different rows milti hain to automatically choose nahi karte; notebook save se pehle rukti hai.


In [15]:
conflicts_found = False
for name, df in datasets.items():
    id_column = "route" if name == "Bus" else "station_id"
    conflicts = int(df.duplicated(subset=[id_column, "date"]).sum())
    print(f"{name}: repeated ID + date rows: {conflicts}")
    if conflicts:
        conflicts_found = True

if conflicts_found:
    raise ValueError("Same ID aur date ki different rows mili hain. Pehle investigate karein.")
ready_to_save = True


Bus: repeated ID + date rows: 0
Rail: repeated ID + date rows: 0


## 8. Cleaned CSVs save karein

Ye **do alag output files** hain: `bus_cleaned.csv` aur `rail_cleaned.csv`. Notebook dobara run karne par ye dono cleaned files overwrite ho jayengi; agar inmein manual changes hon to pehle backup bana lein. Input CSVs overwrite nahi hotin.


In [16]:
if not globals().get("ready_to_save", False):
    raise ValueError("Save se pehle tamam previous cells successfully run karein.")

for name, filename in [("Bus", "bus_cleaned.csv"), ("Rail", "rail_cleaned.csv")]:
    output = data_folder / filename
    datasets[name].to_csv(output, index=False, date_format="%Y-%m-%d")
    print(f"Saved {name}: {output.resolve()} ({len(datasets[name]):,} rows)")


Saved Bus: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\data\processed\bus_cleaned.csv (167,648 rows)
Saved Rail: C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\transit-ridership\backend\data\processed\rail_cleaned.csv (209,472 rows)


## Viva points

- Audit issues identify karta hai; cleaning data ko usable format mein prepare karti hai.
- Exact duplicate rows remove ki, magar conflicting route/station + date records automatically resolve nahi kiye.
- Zero rides retain ki. Missing ya invalid essential values milne par notebook rukti hai; imputation nahi karti.
- Cleaned CSVs alag hain. ML training se related learned preprocessing sirf training data par fit hogi.
